# Evaluation Metrics & Validation Strategies

Choosing the right metric and validation strategy is as important as choosing the right model.

1. **Classification Metrics** - Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC
2. **Regression Metrics** - MSE, RMSE, MAE, R², MAPE
3. **Cross-Validation** - K-Fold, Stratified K-Fold, TimeSeriesSplit
4. **Bias-Variance Tradeoff** - Learning curves and validation curves

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    learning_curve, validation_curve
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")

## 1. Classification Metrics

**Dataset**: Breast Cancer Wisconsin (binary classification)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train a simple model
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, random_state=42)),
])
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Malignant", "Benign"], ax=axes[0], cmap="Blues"
)
axes[0].set_title("Confusion Matrix")

# Classification report as text
print(classification_report(y_test, y_pred, target_names=["Malignant", "Benign"]))

In [ ]:
# ROC Curve and Precision-Recall Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
axes[0].plot(fpr, tpr, color="teal", lw=2, label=f"AUC = {auc_score:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()

# Precision-Recall Curve
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_prob)
ap_score = average_precision_score(y_test, y_prob)
axes[1].plot(recall, precision, color="coral", lw=2, label=f"AP = {ap_score:.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

### When to Use Which Metric

| Metric | Use When | Example |
|--------|---------|--------|
| **Accuracy** | Balanced classes | General classification |
| **Precision** | False positives are costly | Spam detection, fraud alerts |
| **Recall** | False negatives are costly | Cancer detection, security threats |
| **F1 Score** | Need balance of precision/recall | Imbalanced datasets |
| **ROC-AUC** | Compare models, threshold-agnostic | Model selection |
| **PR-AUC** | Heavily imbalanced datasets | Rare event detection |

## 2. Regression Metrics

**Dataset**: California Housing

In [ ]:
housing = fetch_california_housing()
X_h, y_h = housing.data, housing.target
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

reg = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
reg.fit(X_train_h, y_train_h)
y_pred_h = reg.predict(X_test_h)

# Compute metrics
mse = mean_squared_error(y_test_h, y_pred_h)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_h, y_pred_h)
r2 = r2_score(y_test_h, y_pred_h)

print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

In [ ]:
# Predicted vs Actual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_h, y_pred_h, alpha=0.3, s=10, color="teal")
axes[0].plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], "r--", lw=2)
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].set_title("Predicted vs Actual")

# Residual plot
residuals = y_test_h - y_pred_h
axes[1].scatter(y_pred_h, residuals, alpha=0.3, s=10, color="coral")
axes[1].axhline(y=0, color="black", linestyle="--")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual Plot")

plt.tight_layout()
plt.show()

## 3. Cross-Validation Strategies

In [ ]:
# Stratified K-Fold (preserves class proportions in each fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, random_state=42)),
])

# Multiple metrics
for metric in ["accuracy", "roc_auc", "f1"]:
    scores = cross_val_score(pipe_cv, X, y, cv=skf, scoring=metric)
    print(f"{metric:12s}: {scores.mean():.3f} (+/- {scores.std():.3f})  | per fold: {scores.round(3)}")

## 4. Bias-Variance Tradeoff

- **High bias** (underfitting): model too simple, both train and test error are high
- **High variance** (overfitting): model too complex, train error low but test error high
- **Sweet spot**: good balance where test error is minimized

In [ ]:
# Learning Curve - diagnose bias vs variance
train_sizes, train_scores, test_scores = learning_curve(
    pipe_cv, X, y, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring="accuracy"
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
test_mean = test_scores.mean(axis=1)
test_std = test_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="teal")
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="coral")
plt.plot(train_sizes, train_mean, "o-", color="teal", label="Training score")
plt.plot(train_sizes, test_mean, "o-", color="coral", label="Cross-validation score")
plt.xlabel("Training Set Size")
plt.ylabel("Accuracy")
plt.title("Learning Curve (Logistic Regression)")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Validation Curve - effect of hyperparameter on bias/variance
param_range = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

train_scores_vc, test_scores_vc = validation_curve(
    Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=5000))]),
    X, y,
    param_name="clf__C",
    param_range=param_range,
    cv=5, scoring="accuracy", n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.semilogx(param_range, train_scores_vc.mean(axis=1), "o-", color="teal", label="Training")
plt.semilogx(param_range, test_scores_vc.mean(axis=1), "o-", color="coral", label="Cross-validation")
plt.fill_between(param_range, train_scores_vc.mean(axis=1) - train_scores_vc.std(axis=1),
                 train_scores_vc.mean(axis=1) + train_scores_vc.std(axis=1), alpha=0.1, color="teal")
plt.fill_between(param_range, test_scores_vc.mean(axis=1) - test_scores_vc.std(axis=1),
                 test_scores_vc.mean(axis=1) + test_scores_vc.std(axis=1), alpha=0.1, color="coral")
plt.xlabel("Regularization Parameter C")
plt.ylabel("Accuracy")
plt.title("Validation Curve (Logistic Regression)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Accuracy is misleading for imbalanced data** - use F1, PR-AUC, or ROC-AUC instead
2. **Choose metrics based on business cost** - is a false positive or false negative more expensive?
3. **Always use cross-validation** - a single train/test split can be misleading
4. **Learning curves diagnose problems** - converging curves = more data won't help (high bias); diverging = more data might help (high variance)
5. **Validation curves guide tuning** - find the sweet spot before the gap between train/test widens